# Weather Agent (Agent 1)
This agent fetches real-time weather data from OpenWeatherMap API
based on the city name input. It returns structured data including:
- City
- Latitude & Longitude
- Temperature
- Weather Condition
- Humidity


In [10]:
from phi.agent import Agent

In [11]:
# Import required libraries
import os
import requests
from dotenv import load_dotenv

# Load the .env file
load_dotenv()

# Get the OpenWeatherMap API key
API_KEY = os.getenv("WEATHER_API_KEY")

# Optional: check if it loaded correctly
if API_KEY is None:
    raise ValueError("WEATHER_API_KEY not found. Make sure it is set in .env file.")
else:
    print("API key loaded successfully!")


API key loaded successfully!


In [12]:
def weather_api_tool(city_name: str):
    """
    PhiData tool function to fetch structured weather data for a city.
    """
    # Step 1: Geocoding API to get latitude & longitude
    geo_url = f"http://api.openweathermap.org/geo/1.0/direct?q={city_name}&limit=1&appid={API_KEY}"
    geo_response = requests.get(geo_url)
    
    if geo_response.status_code != 200:
        return {"error": f"Geocoding API error: {geo_response.status_code}"}
    
    geo_data = geo_response.json()
    if not geo_data:
        return {"error": f"City '{city_name}' not found."}
    
    lat = geo_data[0]["lat"]
    lon = geo_data[0]["lon"]
    
    # Step 2: Weather API using coordinates
    weather_url = f"http://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&units=metric&appid={API_KEY}"
    weather_response = requests.get(weather_url)
    
    if weather_response.status_code != 200:
        return {"error": f"Weather API error: {weather_response.status_code}"}
    
    weather_data = weather_response.json()
    
    # Step 3: Return structured data
    data = {
        "city": city_name,
        "lat": lat,
        "lon": lon,
        "temperature": weather_data["main"]["temp"],
        "condition": weather_data["weather"][0]["main"],
        "humidity": weather_data["main"]["humidity"]
    }

    return str(data)


In [15]:
weather_agent = Agent(
    name="Weather Agent",
    tools=[weather_api_tool],
    instructions=[
        "You receive a city name and must return structured weather data.",
        "Use the weather_api_tool to fetch real-time data.",
        "Return only factual data (temperature, condition, humidity, coordinates).",
        "Do not generate paragraphs or explanations."
    ]
)
